week3_assignment.ipynb

# Week 3 Assignment -- Customer Intelligence System
## Country Socioeconomic Data -- Classification, Ensemble Learning & Clustering

Pipeline: Data Loading -> Preprocessing -> EDA -> Feature Engineering -> Classification -> Ensemble (RF, XGBoost) -> Clustering (K-Means, DBSCAN)

---


In [ ]:

# setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, confusion_matrix, roc_auc_score, roc_curve)
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

# xgboost might not be installed so wrapping in try
try:
    from xgboost import XGBClassifier
    print("XGBoost imported successfully")
except ImportError:
    import subprocess
    subprocess.check_call(['pip', 'install', 'xgboost'])
    from xgboost import XGBClassifier
    print("XGBoost installed and imported")

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_style('whitegrid')

print("All imports OK")



---
## Part 1 -- Data Loading & Initial Exploration


In [ ]:

# load the dataset
df = pd.read_csv('Country-data.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumn dtypes:\n{df.dtypes}")
print(f"\nFirst 5 rows:")
print(df.head())

# basic info
print(f"\n--- Dataset Info ---")
print(f"Total records: {len(df)}")
print(f"Columns: {list(df.columns)}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nBasic statistics:")
print(df.describe())

# reading data dictionary to understand columns better
dd = pd.read_csv('data-dictionary.csv')
print("\n--- Data Dictionary ---")
print(dd.to_string(index=False))



---
## Part 2 -- Data Preprocessing


In [ ]:

# check duplicates first
dupes = df.duplicated().sum()
print(f"Duplicate rows: {dupes}")
if dupes > 0:
    df = df.drop_duplicates()
    print(f"After removing duplicates: {df.shape}")

# missing values check
print(f"\nMissing values per column:\n{df.isnull().sum()}")

# filling numeric nulls with median if there are any
numeric_cols = df.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f"Filled {col} nulls with median: {median_val}")

print(f"\nTotal nulls remaining: {df.isnull().sum().sum()}")



creating a target variable for classification
basically assigning development status based on income, life expectancy and child mortality
took the thresholds from World Bank classification roughly:
Low income: < 4000, Lower-middle: 4000-10000, Upper-middle: 10000-25000, High: > 25000


In [ ]:

def classify_development(row):
    income = row['income']
    life_exp = row['life_expec']
    child_mort = row['child_mort']

    # using a score based approach, seemed better than just using income alone
    score = 0

    if income > 25000:
        score += 3
    elif income > 10000:
        score += 2
    elif income > 4000:
        score += 1

    if life_exp > 75:
        score += 2
    elif life_exp > 65:
        score += 1

    if child_mort < 15:
        score += 2
    elif child_mort < 50:
        score += 1

    if score >= 6:
        return 'Developed'
    elif score >= 3:
        return 'Developing'
    else:
        return 'Underdeveloped'

df['Development_Status'] = df.apply(classify_development, axis=1)

print("\n--- Target Variable Distribution ---")
print(df['Development_Status'].value_counts())
print(f"\nClass proportions:\n{df['Development_Status'].value_counts(normalize=True).round(3)}")

# encode target
le_target = LabelEncoder()
df['Status_encoded'] = le_target.fit_transform(df['Development_Status'])
print(f"\nLabel mapping: {dict(zip(le_target.classes_, le_target.transform(le_target.classes_)))}")

print(f"\nPreprocessed data shape: {df.shape}")
print(df.head())



---
## Part 3 -- Exploratory Data Analysis (EDA)

### 3.1 Univariate Analysis


In [ ]:

fig, axes = plt.subplots(3, 3, figsize=(16, 14))

feature_names = ['child_mort', 'exports', 'health', 'imports', 'income', 'inflation',
                 'life_expec', 'total_fer', 'gdpp']
colors = ['steelblue', 'coral', 'mediumseagreen', 'mediumpurple', 'goldenrod',
          'indianred', 'teal', 'darkorange', 'slategray']

for idx, (feat, color) in enumerate(zip(feature_names, colors)):
    row, col = idx // 3, idx % 3
    axes[row, col].hist(df[feat], bins=25, color=color, edgecolor='black', alpha=0.7)
    axes[row, col].set_title(f'Distribution of {feat}')
    axes[row, col].set_xlabel(feat)

plt.tight_layout()
plt.show()



### 3.2 Target Class Analysis


In [ ]:
"""### 3.2 Target Class Analysis"""
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# class distribution
status_counts = df['Development_Status'].value_counts()
status_counts.plot(kind='bar', ax=axes[0], color=['mediumseagreen', 'goldenrod', 'indianred'],
                   edgecolor='black', alpha=0.8)
axes[0].set_title('Development Status Distribution')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

# income boxplot by status
status_order = ['Underdeveloped', 'Developing', 'Developed']
box_data = [df[df['Development_Status'] == s]['income'].values for s in status_order]
bp = axes[1].boxplot(box_data, labels=status_order, patch_artist=True)
box_colors = ['indianred', 'goldenrod', 'mediumseagreen']
for patch, color in zip(bp['boxes'], box_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].set_title('Income by Development Status')
axes[1].set_ylabel('Income')

# life expectancy boxplot
box_data2 = [df[df['Development_Status'] == s]['life_expec'].values for s in status_order]
bp2 = axes[2].boxplot(box_data2, labels=status_order, patch_artist=True)
for patch, color in zip(bp2['boxes'], box_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[2].set_title('Life Expectancy by Development Status')
axes[2].set_ylabel('Life Expectancy')

plt.tight_layout()
plt.show()



### 3.3 Bivariate Analysis


In [ ]:
"""### 3.3 Bivariate Analysis"""
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# child mortality vs life expectancy -- should be strongly correlated
for status, color in zip(status_order, box_colors):
    mask = df['Development_Status'] == status
    axes[0].scatter(df[mask]['child_mort'], df[mask]['life_expec'],
                    c=color, label=status, alpha=0.6, edgecolors='k', linewidth=0.5)
axes[0].set_xlabel('Child Mortality')
axes[0].set_ylabel('Life Expectancy')
axes[0].set_title('Child Mortality vs Life Expectancy')
axes[0].legend()

# income vs gdpp
for status, color in zip(status_order, box_colors):
    mask = df['Development_Status'] == status
    axes[1].scatter(df[mask]['income'], df[mask]['gdpp'],
                    c=color, label=status, alpha=0.6, edgecolors='k', linewidth=0.5)
axes[1].set_xlabel('Income')
axes[1].set_ylabel('GDP per capita')
axes[1].set_title('Income vs GDP per Capita')
axes[1].legend()

# fertility vs child mortality
for status, color in zip(status_order, box_colors):
    mask = df['Development_Status'] == status
    axes[2].scatter(df[mask]['total_fer'], df[mask]['child_mort'],
                    c=color, label=status, alpha=0.6, edgecolors='k', linewidth=0.5)
axes[2].set_xlabel('Total Fertility Rate')
axes[2].set_ylabel('Child Mortality')
axes[2].set_title('Fertility Rate vs Child Mortality')
axes[2].legend()

plt.tight_layout()
plt.show()



### 3.4 Correlation Analysis


In [ ]:
"""### 3.4 Correlation Analysis"""
numeric_df = df.select_dtypes(include=[np.number]).drop(columns=['Status_encoded'])
corr_matrix = numeric_df.corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f',
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Correlation Heatmap')
plt.tight_layout()
plt.show()

# finding strongest correlations
print("Top correlations (absolute values):")
upper_corr = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
sorted_corr = upper_corr.unstack().dropna().abs().sort_values(ascending=False)
print(sorted_corr.head(10))



---
## Part 4 -- Feature Engineering


In [ ]:

# health spending efficiency -- how much health spending relative to income
df['Health_Efficiency'] = df['health'] / (df['income'] / 1000 + 1)

# trade balance -- simple exports minus imports
df['Trade_Balance'] = df['exports'] - df['imports']

# gdpp to income ratio
df['GDP_Income_Ratio'] = df['gdpp'] / (df['income'] + 1)

# mortality to fertility -- higher ratio = worse demographic situation
df['Mort_Fertility_Ratio'] = df['child_mort'] / (df['total_fer'] + 0.1)

# log transforms on the really skewed columns
df['Log_Income'] = np.log1p(df['income'])
df['Log_GDPP'] = np.log1p(df['gdpp'])
df['Log_Child_Mort'] = np.log1p(df['child_mort'])

# some interaction features
df['Income_LifeExp'] = df['income'] * df['life_expec']
df['Inflation_Income'] = df['inflation'] * df['income']

print("New features created:")
print(df[['Health_Efficiency', 'Trade_Balance', 'GDP_Income_Ratio',
          'Mort_Fertility_Ratio', 'Log_Income']].head(10))
print(f"\nUpdated shape: {df.shape}")
print(f"All columns: {list(df.columns)}")



### 4.1 Feature Importance (quick look with RF)


In [ ]:
"""### 4.1 Feature Importance (quick look with RF)"""
# throwing all features into a RF to see what matters most
feature_cols_all = ['child_mort', 'exports', 'health', 'imports', 'income', 'inflation',
                    'life_expec', 'total_fer', 'gdpp', 'Health_Efficiency', 'Trade_Balance',
                    'GDP_Income_Ratio', 'Mort_Fertility_Ratio', 'Log_Income', 'Log_GDPP',
                    'Log_Child_Mort', 'Income_LifeExp', 'Inflation_Income']

X_temp = df[feature_cols_all]
y_temp = df['Status_encoded']

rf_temp = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_temp.fit(X_temp, y_temp)

importances = pd.Series(rf_temp.feature_importances_, index=feature_cols_all).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
importances.plot(kind='barh', color='steelblue', ax=ax)
ax.set_title('Feature Importances (Random Forest)')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

print("\nTop 5 features:")
print(importances.tail(5))

# picking features above 0.02 threshold
selected_features = importances[importances > 0.02].index.tolist()
print(f"\nSelected features ({len(selected_features)}): {selected_features}")



---
## Part 5 -- Classification Modeling

### 5.1 Prepare Data


In [ ]:

# these are the features im going with
features = ['child_mort', 'exports', 'health', 'imports', 'income', 'inflation',
            'life_expec', 'total_fer', 'gdpp', 'Health_Efficiency', 'Trade_Balance',
            'GDP_Income_Ratio', 'Mort_Fertility_Ratio', 'Log_Income', 'Log_GDPP',
            'Log_Child_Mort', 'Income_LifeExp']

X = df[features]
y = df['Status_encoded']

# stratified split so each class is represented properly
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                     random_state=42, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"\nTrain class distribution:\n{pd.Series(y_train).value_counts().sort_index()}")
print(f"\nTest class distribution:\n{pd.Series(y_test).value_counts().sort_index()}")

# scaling for models that need it
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)



### 5.2 Train Multiple Classification Models


In [ ]:
"""### 5.2 Train Multiple Classification Models"""
def evaluate_classifier(name, model, X_tr, X_te, y_tr, y_te, needs_scaling=True):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)

    acc = accuracy_score(y_te, y_pred)
    prec = precision_score(y_te, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_te, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_te, y_pred, average='weighted', zero_division=0)

    print(f"\n--- {name} ---")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1 Score:  {f1:.4f}")

    return {'name': name, 'accuracy': acc, 'precision': prec, 'recall': rec,
            'f1': f1, 'predictions': y_pred, 'model': model}

results = []

# 1. logistic regression -- needs scaled data
res = evaluate_classifier("Logistic Regression",
                          LogisticRegression(max_iter=1000, random_state=42, multi_class='multinomial'),
                          X_train_scaled, X_test_scaled, y_train, y_test)
results.append(res)

# 2. decision tree
res = evaluate_classifier("Decision Tree",
                          DecisionTreeClassifier(max_depth=5, random_state=42),
                          X_train, X_test, y_train, y_test)
results.append(res)

# 3. KNN -- also needs scaling
res = evaluate_classifier("KNN (k=5)",
                          KNeighborsClassifier(n_neighbors=5),
                          X_train_scaled, X_test_scaled, y_train, y_test)
results.append(res)

# 4. SVM
res = evaluate_classifier("SVM (RBF)",
                          SVC(kernel='rbf', probability=True, random_state=42),
                          X_train_scaled, X_test_scaled, y_train, y_test)
results.append(res)

# 5. random forest -- tree based so doesnt need scaling
res = evaluate_classifier("Random Forest",
                          RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1),
                          X_train, X_test, y_train, y_test)
results.append(res)

# 6. gradient boosting
res = evaluate_classifier("Gradient Boosting",
                          GradientBoostingClassifier(n_estimators=200, max_depth=5, random_state=42),
                          X_train, X_test, y_train, y_test)
results.append(res)

# 7. XGBoost
res = evaluate_classifier("XGBoost",
                          XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1,
                                        random_state=42, eval_metric='mlogloss', verbosity=0),
                          X_train, X_test, y_train, y_test)
results.append(res)



### 5.3 Model Comparison


In [ ]:
"""### 5.3 Model Comparison"""
results_df = pd.DataFrame(results)[['name', 'accuracy', 'precision', 'recall', 'f1']]
print("\n=== Model Comparison ===")
print(results_df.to_string(index=False))

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
names = results_df['name']
colors = ['#4C72B0', '#55A868', '#C44E52', '#8172B2', '#CCB974', '#64B5CD', '#DD8452']

metrics = ['accuracy', 'precision', 'recall', 'f1']
titles = ['Accuracy', 'Precision', 'Recall', 'F1 Score']

for i, (metric, title) in enumerate(zip(metrics, titles)):
    axes[i].barh(names, results_df[metric], color=colors)
    axes[i].set_title(f'{title} (higher is better)')
    axes[i].set_xlabel(title)
    axes[i].set_xlim(0, 1.05)

plt.tight_layout()
plt.show()

best = results_df.loc[results_df['f1'].idxmax()]
print(f"\nBest model: {best['name']} with F1 = {best['f1']:.4f}")



### 5.4 Confusion Matrix -- Best Models


In [ ]:
"""### 5.4 Confusion Matrix -- Best Models"""
# showing confusion matrices for top 3
top3_names = results_df.nlargest(3, 'f1')['name'].tolist()
top3_results = [r for r in results if r['name'] in top3_names]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, res in enumerate(top3_results):
    cm = confusion_matrix(y_test, res['predictions'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=le_target.classes_, yticklabels=le_target.classes_)
    axes[idx].set_title(f"{res['name']}\n(F1={res['f1']:.3f})")
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')

plt.tight_layout()
plt.show()

best_result = [r for r in results if r['name'] == best['name']][0]
print(f"\n=== Detailed Report: {best['name']} ===")
print(classification_report(y_test, best_result['predictions'],
                           target_names=le_target.classes_))



---
## Part 6 -- Ensemble Learning (Random Forest & XGBoost)

### 6.1 Hyperparameter Tuning -- Random Forest


In [ ]:

# grid search for RF -- kept the grid reasonable so it doesnt run forever
param_grid_rf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 7, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

rf_model = RandomForestClassifier(random_state=42, n_jobs=-1)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_rf = GridSearchCV(rf_model, param_grid_rf, cv=skf, scoring='f1_weighted',
                       n_jobs=-1, verbose=1)
grid_rf.fit(X_train, y_train)

print(f"\nBest RF parameters: {grid_rf.best_params_}")
print(f"Best CV F1 score: {grid_rf.best_score_:.4f}")

best_rf = grid_rf.best_estimator_
y_pred_rf = best_rf.predict(X_test)

print(f"\n--- Tuned Random Forest (Test Set) ---")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_rf, average='weighted'):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_rf, average='weighted'):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred_rf, average='weighted'):.4f}")



### 6.2 Hyperparameter Tuning -- XGBoost


In [ ]:
"""### 6.2 Hyperparameter Tuning -- XGBoost"""
param_grid_xgb = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'min_child_weight': [1, 3, 5],
    'subsample': [0.8, 1.0]
}

xgb_model = XGBClassifier(random_state=42, eval_metric='mlogloss', verbosity=0)

grid_xgb = GridSearchCV(xgb_model, param_grid_xgb, cv=skf, scoring='f1_weighted',
                        n_jobs=-1, verbose=1)
grid_xgb.fit(X_train, y_train)

print(f"\nBest XGBoost parameters: {grid_xgb.best_params_}")
print(f"Best CV F1 score: {grid_xgb.best_score_:.4f}")

best_xgb = grid_xgb.best_estimator_
y_pred_xgb = best_xgb.predict(X_test)

print(f"\n--- Tuned XGBoost (Test Set) ---")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_xgb):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_xgb, average='weighted'):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_xgb, average='weighted'):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred_xgb, average='weighted'):.4f}")



### 6.3 Voting Ensemble -- Combining Best Models


In [ ]:
"""### 6.3 Voting Ensemble -- Combining Best Models"""
# soft voting -- averages the probabilities from each model
voting_clf = VotingClassifier(
    estimators=[
        ('rf', best_rf),
        ('xgb', best_xgb),
        ('gb', GradientBoostingClassifier(n_estimators=200, max_depth=5, random_state=42))
    ],
    voting='soft'
)

voting_clf.fit(X_train, y_train)
y_pred_ensemble = voting_clf.predict(X_test)

print("\n--- Voting Ensemble (Test Set) ---")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_ensemble):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_ensemble, average='weighted'):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_ensemble, average='weighted'):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred_ensemble, average='weighted'):.4f}")



### 6.4 Cross-Validation Comparison


In [ ]:
"""### 6.4 Cross-Validation Comparison"""
models_cv = {
    'Tuned RF': best_rf,
    'Tuned XGBoost': best_xgb,
    'Voting Ensemble': voting_clf
}

print("\n=== 5-Fold Stratified Cross-Validation ===")
cv_results = {}
for name, model in models_cv.items():
    scores = cross_val_score(model, X_train, y_train, cv=skf, scoring='f1_weighted')
    cv_results[name] = scores
    print(f"{name}: F1 = {scores.mean():.4f} (+/- {scores.std():.4f})")

fig, ax = plt.subplots(figsize=(10, 5))
cv_data = [cv_results[name] for name in cv_results]
bp = ax.boxplot(cv_data, labels=list(cv_results.keys()), patch_artist=True)
for patch, color in zip(bp['boxes'], ['steelblue', 'coral', 'mediumseagreen']):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_title('Cross-Validation F1 Scores Comparison')
ax.set_ylabel('F1 Score (weighted)')
plt.tight_layout()
plt.show()



### 6.5 Feature Importance -- Tuned Models


In [ ]:
"""### 6.5 Feature Importance -- Tuned Models"""
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

rf_importances = pd.Series(best_rf.feature_importances_, index=features).sort_values(ascending=True)
rf_importances.plot(kind='barh', color='steelblue', ax=axes[0])
axes[0].set_title('Feature Importance -- Tuned Random Forest')
axes[0].set_xlabel('Importance')

xgb_importances = pd.Series(best_xgb.feature_importances_, index=features).sort_values(ascending=True)
xgb_importances.plot(kind='barh', color='coral', ax=axes[1])
axes[1].set_title('Feature Importance -- Tuned XGBoost')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.show()

print("\nTop 5 features (Random Forest):")
print(rf_importances.tail(5))
print("\nTop 5 features (XGBoost):")
print(xgb_importances.tail(5))



### 6.6 Ensemble Confusion Matrix & Classification Report


In [ ]:
"""### 6.6 Ensemble Confusion Matrix & Classification Report"""
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

preds_dict = {
    'Tuned Random Forest': y_pred_rf,
    'Tuned XGBoost': y_pred_xgb,
    'Voting Ensemble': y_pred_ensemble
}

for idx, (name, preds) in enumerate(preds_dict.items()):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=le_target.classes_, yticklabels=le_target.classes_)
    f1 = f1_score(y_test, preds, average='weighted')
    axes[idx].set_title(f"{name}\n(F1={f1:.3f})")
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')

plt.tight_layout()
plt.show()

print("\n=== Final Ensemble Classification Report ===")
print(classification_report(y_test, y_pred_ensemble, target_names=le_target.classes_))



---
## Part 7 -- Clustering (K-Means & DBSCAN)

### 7.1 Prepare Clustering Data


In [ ]:

# for clustering we only use the original numeric features, no target variable
cluster_features = ['child_mort', 'exports', 'health', 'imports', 'income',
                    'inflation', 'life_expec', 'total_fer', 'gdpp']

X_cluster = df[cluster_features].copy()

# need to standardize since features are on different scales
scaler_cluster = StandardScaler()
X_cluster_scaled = scaler_cluster.fit_transform(X_cluster)

print(f"Clustering data shape: {X_cluster_scaled.shape}")

# PCA to bring it down to 2D for plotting
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_cluster_scaled)

print(f"PCA explained variance ratio: {pca.explained_variance_ratio_}")
print(f"Total variance explained: {pca.explained_variance_ratio_.sum():.4f}")

# 3D PCA too just to see how much more variance we capture
pca_3d = PCA(n_components=3)
X_pca_3d = pca_3d.fit_transform(X_cluster_scaled)
print(f"\n3D PCA total variance explained: {pca_3d.explained_variance_ratio_.sum():.4f}")



### 7.2 K-Means Clustering -- Finding Optimal K


In [ ]:
"""### 7.2 K-Means Clustering -- Finding Optimal K"""
# trying K from 2 to 10 and checking elbow + silhouette
K_range = range(2, 11)
inertias = []
silhouette_scores = []
calinski_scores = []

for k in K_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, max_iter=300, random_state=42)
    km_labels = km.fit_predict(X_cluster_scaled)
    inertias.append(km.inertia_)
    silhouette_scores.append(silhouette_score(X_cluster_scaled, km_labels))
    calinski_scores.append(calinski_harabasz_score(X_cluster_scaled, km_labels))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method')

axes[1].plot(K_range, silhouette_scores, 'rs-', linewidth=2, markersize=8)
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Analysis')

axes[2].plot(K_range, calinski_scores, 'g^-', linewidth=2, markersize=8)
axes[2].set_xlabel('Number of Clusters (K)')
axes[2].set_ylabel('Calinski-Harabasz Score')
axes[2].set_title('Calinski-Harabasz Index')

plt.tight_layout()
plt.show()

optimal_k = list(K_range)[np.argmax(silhouette_scores)]
print(f"\nOptimal K (max silhouette): {optimal_k}")
print(f"Silhouette score at K={optimal_k}: {max(silhouette_scores):.4f}")



### 7.3 K-Means -- Final Clustering


In [ ]:
"""### 7.3 K-Means -- Final Clustering"""
kmeans = KMeans(n_clusters=optimal_k, init='k-means++', n_init=10,
                max_iter=300, random_state=42)
km_labels = kmeans.fit_predict(X_cluster_scaled)

df['KMeans_Cluster'] = km_labels

print(f"\n--- K-Means Results (K={optimal_k}) ---")
print(f"Silhouette Score: {silhouette_score(X_cluster_scaled, km_labels):.4f}")
print(f"Calinski-Harabasz Score: {calinski_harabasz_score(X_cluster_scaled, km_labels):.4f}")
print(f"Davies-Bouldin Score: {davies_bouldin_score(X_cluster_scaled, km_labels):.4f}")
print(f"\nCluster sizes:\n{pd.Series(km_labels).value_counts().sort_index()}")

# plotting clusters in PCA space
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

scatter1 = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=km_labels, cmap='viridis',
                            alpha=0.7, edgecolors='k', linewidth=0.5, s=60)
centroids_pca = pca.transform(kmeans.cluster_centers_)
axes[0].scatter(centroids_pca[:, 0], centroids_pca[:, 1], c='red', marker='X',
                s=200, edgecolors='black', linewidth=2, label='Centroids')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
axes[0].set_title(f'K-Means Clustering (K={optimal_k})')
axes[0].legend()
plt.colorbar(scatter1, ax=axes[0], label='Cluster')

# compare with actual labels
status_colors = {'Underdeveloped': 0, 'Developing': 1, 'Developed': 2}
color_vals = df['Development_Status'].map(status_colors)
scatter2 = axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=color_vals, cmap='RdYlGn',
                            alpha=0.7, edgecolors='k', linewidth=0.5, s=60)
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
axes[1].set_title('Actual Development Status')
plt.colorbar(scatter2, ax=axes[1], label='Status')

plt.tight_layout()
plt.show()



### 7.4 DBSCAN Clustering


In [ ]:
"""### 7.4 DBSCAN Clustering"""


DBSCAN is density based so it can find weird shaped clusters
first need to figure out good eps value using k-distance graph


In [ ]:
from sklearn.neighbors import NearestNeighbors

k = 5
nn = NearestNeighbors(n_neighbors=k)
nn.fit(X_cluster_scaled)
distances, indices = nn.kneighbors(X_cluster_scaled)

k_distances = np.sort(distances[:, k-1])

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(len(k_distances)), k_distances, color='steelblue', linewidth=1.5)
ax.set_xlabel('Data Points (sorted)')
ax.set_ylabel(f'{k}-th Nearest Neighbor Distance')
ax.set_title(f'K-Distance Graph (k={k})')
ax.axhline(y=3.0, color='red', linestyle='--', alpha=0.7, label='eps ~ 3.0')
ax.legend()
plt.tight_layout()
plt.show()

# trying different eps values to see which works best
eps_values = [2.0, 2.5, 3.0, 3.5, 4.0]
best_eps = None
best_sil = -1
best_db_labels = None

print("\n--- DBSCAN Parameter Search ---")
for eps in eps_values:
    db = DBSCAN(eps=eps, min_samples=5)
    db_labels = db.fit_predict(X_cluster_scaled)
    n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
    n_noise = list(db_labels).count(-1)

    if n_clusters >= 2:
        mask = db_labels != -1
        if mask.sum() > n_clusters:
            sil = silhouette_score(X_cluster_scaled[mask], db_labels[mask])
            print(f"eps={eps}: {n_clusters} clusters, {n_noise} noise points, silhouette={sil:.4f}")
            if sil > best_sil:
                best_sil = sil
                best_eps = eps
                best_db_labels = db_labels.copy()
        else:
            print(f"eps={eps}: {n_clusters} clusters, {n_noise} noise points (too few for silhouette)")
    else:
        print(f"eps={eps}: {n_clusters} clusters, {n_noise} noise points (need >= 2)")

# fallback if nothing worked
if best_eps is None:
    best_eps = 3.0
    db = DBSCAN(eps=best_eps, min_samples=5)
    best_db_labels = db.fit_predict(X_cluster_scaled)

print(f"\nBest eps: {best_eps}")

dbscan = DBSCAN(eps=best_eps, min_samples=5)
db_labels = dbscan.fit_predict(X_cluster_scaled)
df['DBSCAN_Cluster'] = db_labels

n_clusters_db = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise = list(db_labels).count(-1)

print(f"\n--- DBSCAN Results (eps={best_eps}, min_samples=5) ---")
print(f"Number of clusters: {n_clusters_db}")
print(f"Noise points: {n_noise}")
print(f"Cluster distribution:\n{pd.Series(db_labels).value_counts().sort_index()}")

if n_clusters_db >= 2:
    mask = db_labels != -1
    if mask.sum() > 0:
        print(f"Silhouette Score (excl noise): {silhouette_score(X_cluster_scaled[mask], db_labels[mask]):.4f}")

# plot dbscan results
fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=db_labels, cmap='Set1',
                      alpha=0.7, edgecolors='k', linewidth=0.5, s=60)
noise_mask = db_labels == -1
if noise_mask.any():
    ax.scatter(X_pca[noise_mask, 0], X_pca[noise_mask, 1], c='black', marker='x',
               s=80, linewidth=2, label='Noise')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
ax.set_title(f'DBSCAN Clustering (eps={best_eps}, min_samples=5)')
ax.legend()
plt.tight_layout()
plt.show()



### 7.5 K-Means vs DBSCAN Comparison


In [ ]:
"""### 7.5 K-Means vs DBSCAN Comparison"""
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=km_labels, cmap='viridis',
                alpha=0.7, edgecolors='k', linewidth=0.5, s=50)
axes[0].set_title(f'K-Means (K={optimal_k})')
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')

axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=db_labels, cmap='Set1',
                alpha=0.7, edgecolors='k', linewidth=0.5, s=50)
axes[1].set_title(f'DBSCAN (eps={best_eps})')
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')

axes[2].scatter(X_pca[:, 0], X_pca[:, 1], c=color_vals, cmap='RdYlGn',
                alpha=0.7, edgecolors='k', linewidth=0.5, s=50)
axes[2].set_title('Actual Development Status')
axes[2].set_xlabel('PC1'); axes[2].set_ylabel('PC2')

plt.tight_layout()
plt.show()

print("\n=== Clustering Method Comparison ===")
print(f"K-Means: {optimal_k} clusters, Silhouette = {silhouette_score(X_cluster_scaled, km_labels):.4f}")
if n_clusters_db >= 2:
    mask = db_labels != -1
    if mask.sum() > 0:
        print(f"DBSCAN:  {n_clusters_db} clusters + {n_noise} noise, Silhouette (excl noise) = {silhouette_score(X_cluster_scaled[mask], db_labels[mask]):.4f}")



---
## Part 8 -- Customer Segmentation Insights

### 8.1 K-Means Cluster Profiling


In [ ]:

# looking at what each cluster actually represents
print("\n=== K-Means Cluster Profiles ===\n")

cluster_means = df.groupby('KMeans_Cluster')[cluster_features].mean().round(2)
print(cluster_means.T)

# trying to name the clusters based on their characteristics
print("\n--- Cluster Interpretations ---")
for c in range(optimal_k):
    cluster_data = df[df['KMeans_Cluster'] == c]
    mean_income = cluster_data['income'].mean()
    mean_life = cluster_data['life_expec'].mean()
    mean_mort = cluster_data['child_mort'].mean()
    mean_gdpp = cluster_data['gdpp'].mean()
    count = len(cluster_data)

    if mean_income > 20000 and mean_life > 75:
        label = "High-Income Developed"
    elif mean_income > 8000 and mean_life > 68:
        label = "Upper-Middle Income"
    elif mean_income > 3000 and mean_life > 60:
        label = "Lower-Middle Income"
    else:
        label = "Low-Income / At-Risk"

    print(f"\nCluster {c} -- '{label}' ({count} countries)")
    print(f"  Avg Income: ${mean_income:,.0f} | Life Exp: {mean_life:.1f} yrs | Child Mort: {mean_mort:.1f}")
    print(f"  Avg GDP/capita: ${mean_gdpp:,.0f}")
    print(f"  Countries: {', '.join(cluster_data['country'].head(5).tolist())}...")



### 8.2 Cluster Visualization -- Radar Chart


In [ ]:
"""### 8.2 Cluster Visualization -- Radar Chart"""
from matplotlib.patches import Circle

# normalize the cluster means so they fit on a radar chart nicely
cluster_means_norm = cluster_means.copy()
for col in cluster_features:
    min_val = cluster_means_norm[col].min()
    max_val = cluster_means_norm[col].max()
    if max_val > min_val:
        cluster_means_norm[col] = (cluster_means_norm[col] - min_val) / (max_val - min_val)
    else:
        cluster_means_norm[col] = 0.5

angles = np.linspace(0, 2 * np.pi, len(cluster_features), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

radar_colors = ['steelblue', 'coral', 'mediumseagreen', 'mediumpurple', 'goldenrod']

for c in range(optimal_k):
    values = cluster_means_norm.loc[c].tolist()
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=f'Cluster {c}',
            color=radar_colors[c % len(radar_colors)])
    ax.fill(angles, values, alpha=0.1, color=radar_colors[c % len(radar_colors)])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(cluster_features, size=9)
ax.set_title('Cluster Profiles -- Radar Chart', size=14, y=1.08)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.show()



### 8.3 Cluster Heatmap


In [ ]:
"""### 8.3 Cluster Heatmap"""
fig, ax = plt.subplots(figsize=(12, 6))

# z-scoring across clusters to see relative differences
cluster_zscore = cluster_means.apply(lambda x: (x - x.mean()) / x.std(), axis=0)
sns.heatmap(cluster_zscore, annot=True, cmap='RdYlGn', center=0, fmt='.2f',
            linewidths=0.5, ax=ax)
ax.set_title('Cluster Profiles -- Z-Score Heatmap')
ax.set_xlabel('Features')
ax.set_ylabel('Cluster')
plt.tight_layout()
plt.show()



### 8.4 Cross-tabulation: Clusters vs Development Status


In [ ]:
"""### 8.4 Cross-tabulation: Clusters vs Development Status"""
# checking how well our unsupervised clusters match the labeled development status
cross_tab = pd.crosstab(df['KMeans_Cluster'], df['Development_Status'], margins=True)
print("\n=== K-Means Clusters vs Development Status ===")
print(cross_tab)

cross_pct = pd.crosstab(df['KMeans_Cluster'], df['Development_Status'], normalize='index').round(3) * 100
print("\n--- Percentage distribution within each cluster ---")
print(cross_pct)

fig, ax = plt.subplots(figsize=(10, 6))
cross_pct.plot(kind='bar', stacked=True, ax=ax,
               color=['mediumseagreen', 'goldenrod', 'indianred'])
ax.set_title('Cluster Composition by Development Status')
ax.set_xlabel('K-Means Cluster')
ax.set_ylabel('Percentage (%)')
ax.legend(title='Status')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("Week 3 Assignment Complete!")
print("="*60)
